# LeetCode 1080 — Insufficient Nodes in Root to Leaf Paths bằng Python

Link đề: https://leetcode.com/problems/insufficient-nodes-in-root-to-leaf-paths/ · Độ khó: **Medium** · Chủ đề: Tree (cây), DFS (duyệt theo chiều sâu), Binary Tree (cây nhị phân)

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def sufficientSubset(self, root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
        ...
```

Ghi chú: cho gốc `root` của một cây nhị phân và số nguyên `limit`. Một nút là **không đủ** (insufficient) nếu **mọi** đường đi từ gốc đến lá đi qua nút đó đều có tổng **nhỏ hơn hẳn** `limit`. Hãy xóa **đồng thời** mọi nút không đủ và trả về gốc của cây còn lại (có thể là cây rỗng, tức `None`). **Lá** là nút không có con. Ý tưởng chính: một nút bị xóa khi và chỉ khi **mọi lá** trong cây con của nó đều có đường đi từ gốc với tổng `< limit`. Dùng DFS "hậu thứ tự": xử lý các con trước; lá bị xóa nếu tổng đường đi `< limit`; nút trong bị xóa nếu **cả hai con** (nếu có) đều đã bị xóa. Kết quả là cây có đáp án duy nhất nên ta so sánh trực tiếp bằng dạng danh sách theo từng tầng của LeetCode.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: định nghĩa **nút không đủ**, vì sao nếu một nút không đủ thì **cả cây con của nó** cũng không đủ, và vì sao khi mọi con bị xóa thì nút trong cũng bị xóa.
2. Tự dựng được lớp `TreeNode`, hàm **dựng cây** từ danh sách theo tầng kiểu LeetCode (`[1,2,3,null,4]`) và hàm **xuất cây** ngược lại, để test mà không phụ thuộc môi trường LeetCode.
3. Nắm mẫu **"hàm đệ quy trả về gốc mới của cây con"** (`node.left = dfs(node.left)`), cách truyền **phần còn lại của `limit`** xuống dưới và cách chuyển DFS đệ quy sang **vòng lặp** (tránh `RecursionError`).
4. Đi qua 4 cách: liệt kê mọi đường đi rồi dựng cây mới (brute force, dùng làm oracle), DFS đệ quy, duyệt theo tầng rồi xử lý ngược (cách nộp chính, không đệ quy) và tính tổng đường đi lớn nhất đi qua mỗi nút.
5. Tránh các lỗi kinh điển: chỉ xóa lá, nhầm `<` với `<=`, chỉ xét tổng từ gốc đến nút hiện tại, xóa nút khi chỉ một con bị xóa, quên trả `None` khi cả gốc bị xóa, và đệ quy quá sâu trên cây lệch.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân và số nguyên `limit`, hãy xóa **đồng thời** mọi **nút không đủ** (insufficient nodes) trong cây và trả về gốc của cây nhị phân thu được.

Một nút là **không đủ** nếu **mọi** đường đi từ gốc đến lá giao với nút này đều có tổng **nhỏ hơn hẳn** `limit`.

**Lá** là nút không có con.

**Ví dụ 1:** `root = [1,2,3,4,-99,-99,7,8,9,-99,-99,12,13,-99,14]`, `limit = 1` -> `[1,2,3,4,null,null,7,8,9,null,14]`.

**Ví dụ 2:** `root = [5,4,8,11,null,17,4,7,1,null,null,5,3]`, `limit = 22` -> `[5,4,8,11,null,17,4,7,null,null,null,5]`.

**Ví dụ 3:** `root = [1,2,-3,-5,null,4,null]`, `limit = -1` -> `[1,null,-3,4]`.

(Cây được cho theo thứ tự từng tầng của LeetCode: mỗi nút không rỗng liệt kê hai con liên tiếp, `null` là con vắng mặt; các `null` ở cuối bị bỏ.)

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[1, 5000]`.
- `-10^5 <= Node.val <= 10^5`
- `-10^9 <= limit <= 10^9`

In [1]:
from typing import Optional, List
from collections import deque
import random, sys, time
from itertools import product

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách theo từng tầng kiểu LeetCode (None là nút vắng mặt)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Xuất cây thành danh sách theo từng tầng kiểu LeetCode (bỏ các None ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

examples = [
    ([1, 2, 3, 4, -99, -99, 7, 8, 9, -99, -99, 12, 13, -99, 14], 1, [1, 2, 3, 4, None, None, 7, 8, 9, None, 14]),
    ([5, 4, 8, 11, None, 17, 4, 7, 1, None, None, 5, 3], 22, [5, 4, 8, 11, None, 17, 4, 7, None, None, None, 5]),
    ([1, 2, -3, -5, None, 4, None], -1, [1, None, -3, 4]),
]
for tree, limit, expected in examples:
    trimmed = list(tree)
    while trimmed and trimmed[-1] is None:                 # danh sách chuẩn không có None ở cuối (ví dụ 3 của đề có một None thừa)
        trimmed.pop()
    assert to_list(build_tree(tree)) == trimmed          # dựng rồi xuất phải trả lại đúng danh sách chuẩn
    print(f"root={tree} limit={limit} -> kỳ vọng {expected}")

root=[1, 2, 3, 4, -99, -99, 7, 8, 9, -99, -99, 12, 13, -99, 14] limit=1 -> kỳ vọng [1, 2, 3, 4, None, None, 7, 8, 9, None, 14]
root=[5, 4, 8, 11, None, 17, 4, 7, 1, None, None, 5, 3] limit=22 -> kỳ vọng [5, 4, 8, 11, None, 17, 4, 7, None, None, None, 5]
root=[1, 2, -3, -5, None, 4, None] limit=-1 -> kỳ vọng [1, None, -3, 4]


## 3. Kiến thức nền

### 3.1. Cây nhị phân và danh sách theo tầng

Một nút `TreeNode` có `val`, `left`, `right` (con trái, con phải; `None` nếu vắng). LeetCode biểu diễn cây bằng danh sách duyệt theo **từng tầng** (level order), trong đó mỗi nút **không rỗng** liệt kê đủ hai con (có thể là `null`). Hàm `build_tree` và `to_list` ở trên chuyển qua lại; hai hàm này dùng hàng đợi nên **không đệ quy** (an toàn với cây sâu `5000`).

In [2]:
tree = build_tree([1, 2, 3, None, 4])
print(tree.val, tree.left.val, tree.right.val, tree.left.left, tree.left.right.val)
print(to_list(tree))
print(to_list(build_tree([1, None, 2, 3])))
print(to_list(None), to_list(build_tree([])))

1 2 3 None 4
[1, 2, 3, None, 4]
[1, None, 2, 3]
[] []


### 3.2. Đường đi từ gốc đến lá và tổng

Một đường đi từ gốc đến lá là dãy nút đi xuống qua các con; **tổng** là tổng các `val` trên đường. Với mỗi nút `v`, các đường đi từ gốc đến lá **đi qua `v`** là các đường đi tới các lá nằm trong cây con của `v`.

In [3]:
def all_root_leaf_paths(root):
    paths, stack = [], [(root, [root.val])]
    while stack:
        node, path = stack.pop()
        if not node.left and not node.right:
            paths.append(path)
        for ch in (node.left, node.right):
            if ch:
                stack.append((ch, path + [ch.val]))
    return sorted(paths)

root = build_tree([5, 4, 8, 11, None, 17, 4, 7, 1, None, None, 5, 3])
for p in all_root_leaf_paths(root):
    print(p, "tổng =", sum(p))

[5, 4, 11, 1] tổng = 21
[5, 4, 11, 7] tổng = 27
[5, 8, 4, 3] tổng = 20
[5, 8, 4, 5] tổng = 22
[5, 8, 17] tổng = 30


### 3.3. Mẫu "hàm đệ quy trả về gốc mới của cây con"

Để xóa nút trong cây, ta hay viết `dfs(node)` **trả về gốc mới** của cây con sau khi xóa (có thể là `None`), rồi gán lại `node.left = dfs(node.left)`, `node.right = dfs(node.right)`. Việc "xóa" một nút thực chất là cha của nó **không còn trỏ** tới nút đó nữa.

In [4]:
def drop_negative_leaves(node):
    """Ví dụ nhỏ: xóa lá có giá trị âm; nút trong mất hết con thì giữ nguyên."""
    if node is None:
        return None
    node.left = drop_negative_leaves(node.left)
    node.right = drop_negative_leaves(node.right)
    if not node.left and not node.right and node.val < 0:
        return None
    return node

t = build_tree([1, -2, 3, -4, 5])
print(to_list(drop_negative_leaves(t)))

[1, -2, 3, None, 5]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `root = [1,2,-3,-5,null,4,null]`, `limit = -1`. Cây: `1` có con trái `2` (con trái `-5`) và con phải `-3` (con trái `4`).

- Đường `1 -> 2 -> -5` có tổng `-2 < -1` ⇒ lá `-5` thuộc nút không đủ; nút `2` chỉ nằm trên đường này nên cũng không đủ.
- Đường `1 -> -3 -> 4` có tổng `2 >= -1` ⇒ các nút `1`, `-3`, `4` đều "đủ" (có ít nhất một đường dài đủ đi qua).
- Xóa `-5` và `2` ⇒ cây `[1, null, -3, 4]` ✓.

Quan sát then chốt:

1. Nút `v` bị xóa khi **mọi** lá trong cây con của `v` có đường từ gốc tổng `< limit`.
2. Nếu `v` bị xóa thì **mọi nút trong cây con** của `v` cũng bị xóa (đường đi qua nút con ⊂ đường đi qua `v`).
3. Vì vậy nút trong bị xóa **khi và chỉ khi mọi con của nó bị xóa**; còn lá bị xóa khi tổng đường từ gốc `< limit`.

## 5. Cách 1 — Brute Force: liệt kê mọi đường đi rồi dựng cây mới

**Ý tưởng.** Đúng nguyên văn đề: liệt kê **mọi đường đi từ gốc đến lá** (danh sách các nút), tính tổng. Một nút được **giữ** nếu có **ít nhất một** đường đi qua nút đó với tổng `>= limit` (nghĩa là nút **không** không-đủ). Cuối cùng dựng một **cây mới** (sao chép) chỉ gồm các nút được giữ. Không sửa cây gốc và không dùng đệ quy trả về, nên là **oracle độc lập**.

In [5]:
def suff_brute(root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
    if root is None:
        return None
    keep = set()                                          # id của các nút có ít nhất một đường dài đủ đi qua
    stack = [(root, [root], root.val)]
    while stack:
        node, path, total = stack.pop()
        if not node.left and not node.right:
            if total >= limit:
                keep.update(id(x) for x in path)
        for ch in (node.left, node.right):
            if ch:
                stack.append((ch, path + [ch], total + ch.val))

    if id(root) not in keep:                              # gốc không đủ: cây rỗng
        return None
    new_root = TreeNode(root.val)
    stack = [(root, new_root)]                            # sao chép lặp (không đệ quy) các nút được giữ
    while stack:
        old_node, new_node = stack.pop()
        if old_node.left is not None and id(old_node.left) in keep:
            new_node.left = TreeNode(old_node.left.val)
            stack.append((old_node.left, new_node.left))
        if old_node.right is not None and id(old_node.right) in keep:
            new_node.right = TreeNode(old_node.right.val)
            stack.append((old_node.right, new_node.right))
    return new_root

for tree, limit, expected in examples:
    got = to_list(suff_brute(build_tree(tree), limit))
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = [1, 2, 3, 4, None, None, 7, 8, 9, None, 14] OK
brute = [5, 4, 8, 11, None, 17, 4, 7, None, None, None, 5] OK
brute = [1, None, -3, 4] OK


**Độ phức tạp và nút thắt.** Có thể có `O(n)` lá và mỗi đường dài `O(n)` (cây "xương cá": một thân dài, mỗi nút thân có thêm một lá) ⇒ `O(n^2)` thời gian và bộ nhớ khi lưu đường đi. Đo thử khi `n` gấp đôi, thời gian gấp khoảng bốn:

In [6]:
def chain_tree(n, vals=None, right=False):
    root = TreeNode(1)
    cur = root
    for i in range(1, n):
        nxt = TreeNode(1 if vals is None else vals[i])
        if right:
            cur.right = nxt
        else:
            cur.left = nxt
        cur = nxt
    return root

def fishbone_tree(n):
    root = TreeNode(1)
    cur = root
    for _ in range(n // 2 - 1):
        cur.right = TreeNode(1)
        cur.left = TreeNode(1)
        cur = cur.left
    return root

for n in (400, 800, 1600, 3200):
    t0 = time.perf_counter()
    suff_brute(fishbone_tree(n), 1)
    dt = time.perf_counter() - t0
    print(f"cây xương cá n={n:4}: {dt * 1000:8.1f} ms")

cây xương cá n= 400:      2.7 ms
cây xương cá n= 800:      7.2 ms
cây xương cá n=1600:     30.4 ms


cây xương cá n=3200:    142.2 ms


Nút thắt: ta lưu lại và duyệt lại cả đường đi cho mỗi lá. Chỉ cần mang theo **tổng đường đi** khi đi xuống và dùng **kết quả của các con** khi đi lên.

## 6. Ý tưởng thuật toán / bước đột phá

**Truyền tổng xuống, truyền kết quả lên.**

- Đi xuống: mang theo `s` = tổng các giá trị từ gốc đến nút hiện tại.
- Tại **lá**: nếu `s < limit` thì lá bị xóa (trả về `None`), ngược lại giữ.
- Tại **nút trong**: gọi đệ quy cho từng con **có mặt**; gán lại con thành kết quả (cây con đã được xử lý). Nếu sau đó **cả hai con đều là `None`** thì nút trong này cũng bị xóa; ngược lại giữ.

Lưu ý nút trong có thể có **một** con: con vắng mặt vốn là `None`, vẫn đúng quy tắc "xóa khi không còn con nào sống".

**Tránh đệ quy sâu.** Cây có thể lệch dài `5000` nên DFS đệ quy dễ gặp `RecursionError` ở Python (giới hạn mặc định ~`1000`). Có thể nâng giới hạn, hoặc duyệt theo **tầng** (BFS) để ghi nhận thứ tự và tổng tiền tố rồi xử lý **ngược lại từ lá lên**.

| Cách | Ý tưởng |
|---|---|
| 2 | DFS đệ quy, truyền tổng xuống, gán lại con khi trở về |
| 3 | BFS ghi tổng tiền tố, xử lý ngược từ lá lên bằng cờ `keep` (nộp) |
| 4 | Tính cho mỗi nút "tổng đường lớn nhất đi qua nó" (tiền tố + gain xuống), xóa nút nào có giá trị `< limit` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS đệ quy (truyền tổng xuống)

**Giải thích từng đoạn.**

- `dfs(node, s)` trả về gốc mới của cây con; `s` là tổng các giá trị **phía trên** `node` (chưa gồm `node`).
- `s += node.val`.
- Lá: `None` nếu `s < limit`, ngược lại `node`.
- Nút trong: `node.left = dfs(node.left, s)` (nếu có), `node.right = dfs(node.right, s)`; nếu cả hai đều `None` ⇒ `None`, ngược lại `node`.

In [7]:
def suff_recursive(root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
    def dfs(node, s):
        if node is None:
            return None
        s += node.val
        if not node.left and not node.right:               # lá
            return None if s < limit else node
        node.left = dfs(node.left, s)
        node.right = dfs(node.right, s)
        return node if (node.left or node.right) else None

    old = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old, 20000))                  # cây lệch tới 5000 tầng
    try:
        return dfs(root, 0)
    finally:
        sys.setrecursionlimit(old)

for tree, limit, expected in examples:
    got = to_list(suff_recursive(build_tree(tree), limit))
    print(f"DFS đệ quy = {got}", "OK" if got == expected else "SAI")
    assert got == expected

DFS đệ quy = [1, 2, 3, 4, None, None, 7, 8, 9, None, 14] OK
DFS đệ quy = [5, 4, 8, 11, None, 17, 4, 7, None, None, None, 5] OK
DFS đệ quy = [1, None, -3, 4] OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def sufficientSubset(self, root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
        import sys
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))     # cây lệch sâu tới 5000

        def dfs(node: Optional[TreeNode], s: int) -> Optional[TreeNode]:
            if node is None:
                return None
            s += node.val                                   # tổng từ gốc tới node
            if not node.left and not node.right:            # lá: giữ nếu tổng đủ lớn
                return None if s < limit else node
            node.left = dfs(node.left, s)                   # xử lý hai cây con trước
            node.right = dfs(node.right, s)
            return node if (node.left or node.right) else None   # còn con nào sống thì giữ

        return dfs(root, 0)

sol_rec = lambda tree, limit: Solution().sufficientSubset(tree, limit)
for tree, limit, expected in examples:
    assert to_list(sol_rec(build_tree(tree), limit)) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n)`, bộ nhớ `O(h)` ngăn xếp đệ quy với `h` là chiều cao (tới `n` khi cây lệch).

---

### 7.2. Cách 3 — BFS ghi tổng tiền tố rồi xử lý ngược (cách nộp chính)

**Giải thích từng đoạn.**

- Duyệt theo tầng, lưu danh sách `nodes` và `prefix[i]` = tổng từ gốc đến `nodes[i]`.
- Duyệt **ngược** danh sách: đảm bảo mọi con được xử lý **trước** cha (con luôn đứng sau cha trong thứ tự BFS).
- Nếu `nodes[i]` là lá: `keep = prefix[i] >= limit`.
- Ngược lại: nếu `left` có mà `keep[left]` sai thì đặt `left = None` (tương tự `right`); `keep[node]` đúng khi còn ít nhất một con.
- Trả về `root` nếu `keep[root]`, ngược lại `None`.

In [9]:
def suff_bfs(root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
    nodes, prefix = [root], [root.val]
    i = 0
    while i < len(nodes):
        node = nodes[i]
        for ch in (node.left, node.right):
            if ch:
                nodes.append(ch)
                prefix.append(prefix[i] + ch.val)
        i += 1
    keep = {}
    for i in range(len(nodes) - 1, -1, -1):                # con luôn đứng sau cha trong thứ tự BFS
        node = nodes[i]
        if not node.left and not node.right:
            keep[node] = prefix[i] >= limit
        else:
            if node.left and not keep[node.left]:
                node.left = None
            if node.right and not keep[node.right]:
                node.right = None
            keep[node] = bool(node.left or node.right)
    return root if keep[root] else None

for tree, limit, expected in examples:
    got = to_list(suff_bfs(build_tree(tree), limit))
    print(f"BFS = {got}", "OK" if got == expected else "SAI")
    assert got == expected

BFS = [1, 2, 3, 4, None, None, 7, 8, 9, None, 14] OK
BFS = [5, 4, 8, 11, None, 17, 4, 7, None, None, None, 5] OK
BFS = [1, None, -3, 4] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def sufficientSubset(self, root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
        nodes = [root]                                      # các nút theo thứ tự BFS
        prefix = [root.val]                                 # prefix[i]: tổng từ gốc đến nodes[i]
        i = 0
        while i < len(nodes):
            node = nodes[i]
            for child in (node.left, node.right):
                if child:
                    nodes.append(child)
                    prefix.append(prefix[i] + child.val)
            i += 1
        keep = {}                                           # keep[node]: nút còn sống (có đường đủ dài đi qua)
        for i in range(len(nodes) - 1, -1, -1):             # con đứng sau cha: xử lý con trước
            node = nodes[i]
            if not node.left and not node.right:            # lá: giữ nếu tổng đủ lớn
                keep[node] = prefix[i] >= limit
            else:
                if node.left and not keep[node.left]:       # cắt con đã bị xóa
                    node.left = None
                if node.right and not keep[node.right]:
                    node.right = None
                keep[node] = bool(node.left or node.right)  # còn con nào sống thì giữ
        return root if keep[root] else None

sol_bfs = lambda tree, limit: Solution().sufficientSubset(tree, limit)
for tree, limit, expected in examples:
    assert to_list(sol_bfs(build_tree(tree), limit)) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n)`, bộ nhớ `O(n)` (danh sách nút, tổng tiền tố, `keep`); không đệ quy nên an toàn với cây lệch `5000`.

---

### 7.3. Cách 4 — Tổng đường lớn nhất đi qua mỗi nút

**Ý tưởng khác.** Với mỗi nút `v`, `through(v)` là **tổng lớn nhất** của một đường từ gốc đến lá đi qua `v`: `through(v) = prefix(v) + down(v)` với `down(v)` là tổng lớn nhất các giá trị từ con của `v` xuống một lá (`0` nếu `v` là lá). Nút `v` không đủ khi và chỉ khi `through(v) < limit`. Tính `prefix` bằng BFS, `down` bằng duyệt ngược, rồi duyệt xuống và cắt các con có `through < limit`; vì nút không đủ kéo theo cả cây con không đủ nên chỉ cần cắt ở nút đầu tiên.

In [11]:
def suff_through(root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
    nodes, prefix = [root], [root.val]
    i = 0
    while i < len(nodes):
        node = nodes[i]
        for ch in (node.left, node.right):
            if ch:
                nodes.append(ch)
                prefix.append(prefix[i] + ch.val)
        i += 1
    down = {}
    for node in reversed(nodes):
        kids = [ch.val + down[ch] for ch in (node.left, node.right) if ch]
        down[node] = max(kids) if kids else 0
    through = {node: prefix[i] + down[node] for i, node in enumerate(nodes)}
    if through[root] < limit:
        return None
    for node in nodes:                                     # mọi nút trong danh sách còn sống vẫn nằm trong cây
        if node.left and through[node.left] < limit:
            node.left = None
        if node.right and through[node.right] < limit:
            node.right = None
    return root

for tree, limit, expected in examples:
    got = to_list(suff_through(build_tree(tree), limit))
    print(f"through = {got}", "OK" if got == expected else "SAI")
    assert got == expected

through = [1, 2, 3, 4, None, None, 7, 8, 9, None, 14] OK
through = [5, 4, 8, 11, None, 17, 4, 7, None, None, None, 5] OK
through = [1, None, -3, 4] OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def sufficientSubset(self, root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
        nodes = [root]                                      # thứ tự BFS
        prefix = [root.val]                                 # tổng từ gốc đến nodes[i]
        i = 0
        while i < len(nodes):
            node = nodes[i]
            for child in (node.left, node.right):
                if child:
                    nodes.append(child)
                    prefix.append(prefix[i] + child.val)
            i += 1
        down = {}                                           # down[v]: tổng lớn nhất từ con của v xuống một lá (0 nếu v là lá)
        for node in reversed(nodes):
            gains = [c.val + down[c] for c in (node.left, node.right) if c]
            down[node] = max(gains) if gains else 0
        through = {node: prefix[i] + down[node] for i, node in enumerate(nodes)}   # tổng lớn nhất của đường qua node
        if through[root] < limit:
            return None
        for node in nodes:                                  # nút có through < limit là nút không đủ: cắt bỏ
            if node.left and through[node.left] < limit:
                node.left = None
            if node.right and through[node.right] < limit:
                node.right = None
        return root

sol_through = lambda tree, limit: Solution().sufficientSubset(tree, limit)
for tree, limit, expected in examples:
    assert to_list(sol_through(build_tree(tree), limit)) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Cách 3 cho ví dụ 3

In thứ tự BFS, tổng tiền tố, và quyết định `keep` khi duyệt ngược.

In [13]:
def trace_bfs(tree_list, limit):
    root = build_tree(tree_list)
    nodes, prefix = [root], [root.val]
    i = 0
    while i < len(nodes):
        node = nodes[i]
        for ch in (node.left, node.right):
            if ch:
                nodes.append(ch)
                prefix.append(prefix[i] + ch.val)
        i += 1
    print("thứ tự BFS:", [n.val for n in nodes], "| tổng tiền tố:", prefix, "| limit =", limit)
    keep = {}
    for i in range(len(nodes) - 1, -1, -1):
        node = nodes[i]
        if not node.left and not node.right:
            keep[node] = prefix[i] >= limit
            print(f"  lá {node.val}: tổng {prefix[i]} {'>=' if keep[node] else '<'} {limit} -> {'giữ' if keep[node] else 'xóa'}")
        else:
            cut = []
            if node.left and not keep[node.left]:
                cut.append(f"trái {node.left.val}")
                node.left = None
            if node.right and not keep[node.right]:
                cut.append(f"phải {node.right.val}")
                node.right = None
            keep[node] = bool(node.left or node.right)
            print(f"  nút {node.val}: cắt {cut or 'không con nào'} -> {'giữ' if keep[node] else 'xóa (hết con)'}")
    result = root if keep[root] else None
    print("kết quả:", to_list(result))
    return to_list(result)

assert trace_bfs([1, 2, -3, -5, None, 4, None], -1) == [1, None, -3, 4]

thứ tự BFS: [1, 2, -3, -5, 4] | tổng tiền tố: [1, 3, -2, -2, 2] | limit = -1
  lá 4: tổng 2 >= -1 -> giữ
  lá -5: tổng -2 < -1 -> xóa
  nút -3: cắt không con nào -> giữ
  nút 2: cắt ['trái -5'] -> xóa (hết con)
  nút 1: cắt ['trái 2'] -> giữ
kết quả: [1, None, -3, 4]


### 8.2. Cách 4 cho ví dụ 2

In `prefix`, `down` và `through` của từng nút; nút nào có `through < limit` bị cắt.

In [14]:
tree_list, limit = [5, 4, 8, 11, None, 17, 4, 7, 1, None, None, 5, 3], 22
root = build_tree(tree_list)
nodes, prefix = [root], [root.val]
i = 0
while i < len(nodes):
    node = nodes[i]
    for ch in (node.left, node.right):
        if ch:
            nodes.append(ch)
            prefix.append(prefix[i] + ch.val)
    i += 1
down = {}
for node in reversed(nodes):
    gains = [c.val + down[c] for c in (node.left, node.right) if c]
    down[node] = max(gains) if gains else 0
print("limit =", limit)
for i, node in enumerate(nodes):
    through = prefix[i] + down[node]
    print(f"nút {node.val:3}: prefix={prefix[i]:3} down={down[node]:3} through={through:3}", "-> KHÔNG ĐỦ (xóa)" if through < limit else "")
assert to_list(suff_through(build_tree(tree_list), limit)) == [5, 4, 8, 11, None, 17, 4, 7, None, None, None, 5]

limit = 22
nút   5: prefix=  5 down= 25 through= 30 
nút   4: prefix=  9 down= 18 through= 27 
nút   8: prefix= 13 down= 17 through= 30 
nút  11: prefix= 20 down=  7 through= 27 
nút  17: prefix= 30 down=  0 through= 30 
nút   4: prefix= 17 down=  5 through= 22 
nút   7: prefix= 27 down=  0 through= 27 
nút   1: prefix= 21 down=  0 through= 21 -> KHÔNG ĐỦ (xóa)
nút   5: prefix= 22 down=  0 through= 22 
nút   3: prefix= 20 down=  0 through= 20 -> KHÔNG ĐỦ (xóa)


### 8.3. Nút trong chỉ còn lại khi có con sống

Cây `[10, 5, -3, 3, 2, None, 11]` với `limit = 18`: lá `2` có đường `10 + 5 + 2 = 17 < 18` bị xóa, nhưng nút `5` vẫn còn vì con `3` (`10 + 5 + 3 = 18`) sống.

In [15]:
tree_list, limit = [10, 5, -3, 3, 2, None, 11], 18
print("trước:", tree_list, "| các đường:", all_root_leaf_paths(build_tree(tree_list)))
print("sau  :", to_list(sol_bfs(build_tree(tree_list), limit)))
assert to_list(sol_bfs(build_tree(tree_list), limit)) == [10, 5, -3, 3, None, None, 11]

trước: [10, 5, -3, 3, 2, None, 11] | các đường: [[10, -3, 11], [10, 5, 2], [10, 5, 3]]
sau  : [10, 5, -3, 3, None, None, 11]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (BFS + xử lý ngược): `O(n)` thời gian, **không đệ quy** nên không lo `RecursionError` với cây lệch sâu `5000`.

```python
class Solution:
    def sufficientSubset(self, root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
        nodes = [root]                                      # các nút theo thứ tự BFS
        prefix = [root.val]                                 # prefix[i]: tổng từ gốc đến nodes[i]
        i = 0
        while i < len(nodes):
            node = nodes[i]
            for child in (node.left, node.right):
                if child:
                    nodes.append(child)
                    prefix.append(prefix[i] + child.val)
            i += 1
        keep = {}                                           # keep[node]: nút còn sống (có đường đủ dài đi qua)
        for i in range(len(nodes) - 1, -1, -1):             # con đứng sau cha: xử lý con trước
            node = nodes[i]
            if not node.left and not node.right:            # lá: giữ nếu tổng đủ lớn
                keep[node] = prefix[i] >= limit
            else:
                if node.left and not keep[node.left]:       # cắt con đã bị xóa
                    node.left = None
                if node.right and not keep[node.right]:
                    node.right = None
                keep[node] = bool(node.left or node.right)  # còn con nào sống thì giữ
        return root if keep[root] else None
```

In [16]:
class Solution:
    def sufficientSubset(self, root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
        nodes = [root]                                      # các nút theo thứ tự BFS
        prefix = [root.val]                                 # prefix[i]: tổng từ gốc đến nodes[i]
        i = 0
        while i < len(nodes):
            node = nodes[i]
            for child in (node.left, node.right):
                if child:
                    nodes.append(child)
                    prefix.append(prefix[i] + child.val)
            i += 1
        keep = {}                                           # keep[node]: nút còn sống (có đường đủ dài đi qua)
        for i in range(len(nodes) - 1, -1, -1):             # con đứng sau cha: xử lý con trước
            node = nodes[i]
            if not node.left and not node.right:            # lá: giữ nếu tổng đủ lớn
                keep[node] = prefix[i] >= limit
            else:
                if node.left and not keep[node.left]:       # cắt con đã bị xóa
                    node.left = None
                if node.right and not keep[node.right]:
                    node.right = None
                keep[node] = bool(node.left or node.right)  # còn con nào sống thì giữ
        return root if keep[root] else None

sol = lambda tree, limit: Solution().sufficientSubset(tree, limit)       # truyền cây dựng mới vì hàm sửa cây tại chỗ
for tree, limit, expected in examples:
    got = to_list(sol(build_tree(tree), limit))
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, None, None, 7, 8, 9, None, 14] OK
[5, 4, 8, 11, None, 17, 4, 7, None, None, None, 5] OK
[1, None, -3, 4] OK


## 10. Vì sao đúng

**Định nghĩa nút giữ.** Gọi `L(v)` là tập các lá trong cây con của `v` (với lá thì `L(v) = {v}`) và `S(l)` là tổng đường từ gốc đến lá `l`. Các đường từ gốc đến lá **đi qua** `v` chính là các đường tới các lá thuộc `L(v)`. Nút `v` **không đủ** khi `S(l) < limit` với mọi `l` thuộc `L(v)`; nút `v` **được giữ** khi tồn tại `l` thuộc `L(v)` có `S(l) >= limit`. Mọi nút "không đủ" bị xóa đồng thời; vì tiêu chí chỉ phụ thuộc cây **ban đầu**, không phụ thuộc thứ tự xóa.

**Hệ quả (quy tắc đệ quy).** Lá `l` được giữ khi và chỉ khi `S(l) >= limit`. Nút trong `v` được giữ khi và chỉ khi **tồn tại một con được giữ**, bởi `L(v)` là hợp của `L(c)` theo các con `c`. Nút bị xóa kéo theo cả cây con bị xóa (mọi nút ở đó cũng không đủ) nên việc "bỏ liên kết từ cha" là đủ để xóa.

**Cách 2 và 3** cài đặt đúng quy tắc trên: lá dựa vào `S`, nút trong dựa vào việc còn con nào sống. Cách 3 dùng thứ tự BFS đảo ngược: mỗi nút xuất hiện **sau** cha trong BFS nên xử lý ngược thì con luôn được quyết định trước cha; thứ tự đúng nên kết quả đúng.

**Cách 4.** `through(v) = prefix(v) + down(v)` là tổng lớn nhất của các đường gốc-lá đi qua `v`: `prefix(v)` cố định phần từ gốc tới `v` và `down(v)` là phần tốt nhất từ `v` xuống lá. `v` không đủ khi tổng lớn nhất cũng `< limit`, tức `through(v) < limit` (đúng định nghĩa). Vì nút không đủ kéo theo cả cây con không đủ, cắt tại nút không đủ đầu tiên (ở cha của nó) là đủ.

**Cây rỗng.** Nếu gốc không đủ thì cây còn lại rỗng, ta trả `None` (xem lỗi 5).

**Kiểm chứng bằng máy.** Oracle (Cách 1) liệt kê mọi đường đi theo định nghĩa và dựng cây mới. Ta đối chiếu mọi cách với oracle trên rất nhiều cây ngẫu nhiên nhỏ (nhiều hình dạng, giá trị âm/dương, nhiều `limit`), và mọi cách khớp nhau trên cây lớn.

In [17]:
def random_tree(rng, n, lo=-10, hi=10, shape="random"):
    """Trả về danh sách theo tầng của một cây ngẫu nhiên có n nút."""
    nodes = [TreeNode(rng.randint(lo, hi))]
    slots = [(0, "L"), (0, "R")]
    for i in range(1, n):
        if shape == "left":
            p, side = i - 1, "L"
        elif shape == "right":
            p, side = i - 1, "R"
        elif shape == "zigzag":
            p, side = i - 1, ("L" if i % 2 else "R")
        else:
            idx = rng.randrange(len(slots))
            slots[idx], slots[-1] = slots[-1], slots[idx]
            p, side = slots.pop()
        node = TreeNode(rng.randint(lo, hi))
        nodes.append(node)
        if side == "L":
            nodes[p].left = node
        else:
            nodes[p].right = node
        if shape == "random":
            slots += [(i, "L"), (i, "R")]
    return to_list(nodes[0])

impls = {"đệ quy": sol_rec, "BFS": sol_bfs, "through": sol_through, "nộp (Cách 3)": sol,
         "suff_recursive": suff_recursive, "suff_bfs": suff_bfs, "suff_through": suff_through}
rng = random.Random(10)
total = 0
for _ in range(4000):
    n = rng.randint(1, 12)
    tree_list = random_tree(rng, n, shape=rng.choice(["random", "random", "left", "right", "zigzag"]))
    limit = rng.randint(-15, 15)
    ref = to_list(suff_brute(build_tree(tree_list), limit))
    for name, fn in impls.items():
        assert to_list(fn(build_tree(tree_list), limit)) == ref, (name, tree_list, limit)
    total += 1
print("đã đối chiếu", total, "cặp (cây, limit) ngẫu nhiên: mọi cách khớp oracle")

đã đối chiếu 4000 cặp (cây, limit) ngẫu nhiên: mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường đi | `O(n^2)` (cây lệch) | `O(n^2)` |
| 2. DFS đệ quy | `O(n)` | `O(h)` ngăn xếp (`h` tới `n`) |
| 3. BFS + xử lý ngược | `O(n)` | `O(n)` |
| 4. Tổng đường lớn nhất qua nút | `O(n)` | `O(n)` |

Đo ở cỡ tối đa (`n = 5000`) với các dạng cây: ngẫu nhiên, lệch trái, lệch phải, zigzag, cân bằng, và các `limit` khác nhau (xóa hết, xóa một phần, không xóa):

In [18]:
rng = random.Random(11)
N = 5000
def complete_tree_list(n, rng):
    return [rng.randint(-10, 10) for _ in range(n)]
shapes = {
    "ngẫu nhiên": random_tree(rng, N),
    "lệch trái": random_tree(rng, N, shape="left"),
    "lệch phải": random_tree(rng, N, shape="right"),
    "zigzag": random_tree(rng, N, shape="zigzag"),
    "cân bằng": complete_tree_list(N, rng),
}
for label, tree_list in shapes.items():
    for limit in (-10 ** 9, 0, 10 ** 9):
        ref = to_list(sol(build_tree(tree_list), limit))
        row = [f"{label:11} limit={limit:11}"]
        for name, fn in (("đệ quy", sol_rec), ("BFS", sol), ("through", sol_through)):
            tree = build_tree(tree_list)
            t0 = time.perf_counter()
            res = fn(tree, limit)
            dt = (time.perf_counter() - t0) * 1000
            assert to_list(res) == ref, (label, limit, name)
            row.append(f"{name} {dt:6.2f} ms")
        print(" | ".join(row), "| số nút còn lại", len([v for v in ref if v is not None]))

ngẫu nhiên  limit=-1000000000 | đệ quy   1.85 ms | BFS   1.95 ms | through   2.01 ms | số nút còn lại 5000
ngẫu nhiên  limit=          0 | đệ quy   2.03 ms | BFS   1.98 ms | through   1.99 ms | số nút còn lại 3779
ngẫu nhiên  limit= 1000000000 | đệ quy   2.02 ms | BFS   1.98 ms | through   1.89 ms | số nút còn lại 0
lệch trái   limit=-1000000000 | đệ quy   1.49 ms | BFS   1.69 ms | through   1.64 ms | số nút còn lại 5000
lệch trái   limit=          0 | đệ quy   1.83 ms | BFS   1.65 ms | through   1.55 ms | số nút còn lại 0
lệch trái   limit= 1000000000 | đệ quy   1.62 ms | BFS   1.59 ms | through   1.56 ms | số nút còn lại 0
lệch phải   limit=-1000000000 | đệ quy   1.84 ms | BFS   1.66 ms | through   1.68 ms | số nút còn lại 5000
lệch phải   limit=          0 | đệ quy   1.85 ms | BFS   1.60 ms | through   1.64 ms | số nút còn lại 0
lệch phải   limit= 1000000000 | đệ quy   1.55 ms | BFS   1.54 ms | through   1.54 ms | số nút còn lại 0
zigzag      limit=-1000000000 | đệ quy   1.52 ms | B

zigzag      limit=          0 | đệ quy   1.75 ms | BFS   1.82 ms | through   1.75 ms | số nút còn lại 5000
zigzag      limit= 1000000000 | đệ quy   2.06 ms | BFS   1.63 ms | through   1.61 ms | số nút còn lại 0
cân bằng    limit=-1000000000 | đệ quy   1.45 ms | BFS   1.59 ms | through   1.63 ms | số nút còn lại 5000
cân bằng    limit=          0 | đệ quy   1.68 ms | BFS   1.64 ms | through   1.63 ms | số nút còn lại 2487


cân bằng    limit= 1000000000 | đệ quy   2.03 ms | BFS   1.60 ms | through   1.80 ms | số nút còn lại 0


## 12. Bộ test

Cây có đáp án **duy nhất**, ta so sánh dạng danh sách theo tầng. Mỗi lần gọi hàm ta **dựng cây mới** từ danh sách vì các cách sửa cây tại chỗ. Bộ test gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm), test ngẫu nhiên có seed (đúng ràng buộc), test cỡ lớn và tính chất.

In [19]:
def check_all(tree_list, limit, expected=None, brute=True):
    results = {name: to_list(fn(build_tree(tree_list), limit)) for name, fn in impls.items()}
    first = results["nộp (Cách 3)"]
    for name, got in results.items():
        assert got == first, (name, tree_list[:15], limit)
    if brute and len([v for v in tree_list if v is not None]) <= 14:
        assert to_list(suff_brute(build_tree(tree_list), limit)) == first, (tree_list, limit)
    if expected is not None:
        assert first == expected, (tree_list, limit, first, expected)
    return first

# 12.1 ví dụ đề
for tree, limit, expected in examples:
    check_all(tree, limit, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([5], 5, [5]),                                       # một nút, tổng bằng limit: giữ (không nhỏ hơn hẳn)
    ([5], 6, []),                                        # một nút, tổng nhỏ hơn limit: cây rỗng
    ([-1], -1, [-1]),                                    # giá trị âm, bằng limit
    ([-1], 0, []),                                       # giá trị âm nhỏ hơn limit
    ([1, 2, 3], 4, [1, None, 3]),                        # đường 1 + 2 = 3 < 4 bị xóa (nút 2); đường 1 + 3 = 4 giữ
    ([1, 2, 3], 5, []),                                  # mọi đường < 5: xóa hết
    ([1, 2, 3], 3, [1, 2, 3]),                           # đường 1 + 2 = 3, 1 + 3 = 4 đều >= 3
    ([1, 2], 3, [1, 2]),                                 # chỉ một con: đường 1 + 2 = 3 >= 3
    ([1, 2], 4, []),                                     # đường 3 < 4: xóa cả hai
    ([1, None, 2], 3, [1, None, 2]),                     # chỉ con phải
    ([1, None, 2], 4, []),
    ([10, 5, -3, 3, 2, None, 11], 18, [10, 5, -3, 3, None, None, 11]),   # lá 2 (đường 17) bị xóa, nút 5 vẫn sống nhờ lá 3
    ([10, 5, -3, 3, 2, None, 11], 17, [10, 5, -3, 3, 2, None, 11]),      # mọi đường >= 17: giữ nguyên
    ([10, 5, -3, 3, 2, None, 11], 19, []),                              # đường lớn nhất 18 < 19: xóa hết
    ([1, 2, 3], -100, [1, 2, 3]),                        # limit rất nhỏ: không xóa gì
    ([1, 2, 3], 10 ** 9, []),                            # limit rất lớn: xóa hết
    ([1, -2, -3], 0, []),                                # mọi đường âm
    ([0, 0, 0, 0, 0, 0, 0], 0, [0, 0, 0, 0, 0, 0, 0]),   # mọi đường 0 == limit: giữ hết
    ([0, 0, 0, 0, 0, 0, 0], 1, []),                      # mọi đường 0 < 1: xóa hết
]
for tree, limit, expected in edge:
    check_all(tree, limit, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 19 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_only_leaves(root, limit):                    # chỉ xóa lá, không xóa nút trong mất hết con
    def dfs(node, s):
        if node is None:
            return None
        s += node.val
        if not node.left and not node.right:
            return None if s < limit else node
        node.left = dfs(node.left, s)
        node.right = dfs(node.right, s)
        return node
    return dfs(root, 0)

def wrong_le(root, limit):                             # dùng <= thay vì < (xóa cả khi tổng bằng limit)
    def dfs(node, s):
        if node is None:
            return None
        s += node.val
        if not node.left and not node.right:
            return None if s <= limit else node
        node.left = dfs(node.left, s)
        node.right = dfs(node.right, s)
        return node if (node.left or node.right) else None
    return dfs(root, 0)

def wrong_prefix_only(root, limit):                    # xóa nút khi tổng từ gốc đến chính nó < limit (bỏ qua phần phía dưới)
    def dfs(node, s):
        if node is None:
            return None
        s += node.val
        if s < limit:
            return None
        node.left = dfs(node.left, s)
        node.right = dfs(node.right, s)
        return node
    return dfs(root, 0)

def wrong_need_both(root, limit):                      # nút trong chỉ giữ khi CẢ HAI con đều sống (con vắng mặt coi như bị xóa)
    def dfs(node, s):
        if node is None:
            return None
        s += node.val
        if not node.left and not node.right:
            return None if s < limit else node
        left = dfs(node.left, s)
        right = dfs(node.right, s)
        if left is None or right is None:
            return None
        node.left, node.right = left, right
        return node
    return dfs(root, 0)

def wrong_keep_root(root, limit):                      # quên trả None khi gốc bị xóa: luôn trả root
    res = sol(root, limit)
    return res if res is not None else root

def find_counter(wrong, tries=40000, seed=9):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        n = rng.randint(1, 7)
        tree_list = random_tree(rng, n, -4, 6, shape=rng.choice(["random", "left", "zigzag"]))
        limit = rng.randint(-5, 8)
        if best is not None and n >= best[2]:
            continue
        if to_list(wrong(build_tree(tree_list), limit)) != to_list(suff_brute(build_tree(tree_list), limit)):
            best = (tree_list, limit, n)
    return best

hand_cases = {                                         # phản ví dụ chọn tay (dễ nhìn), được xác nhận lại bằng oracle
    "chỉ xóa lá, không xóa nút trong": (wrong_only_leaves, [1, 2, -3, -5, None, 4, None], -1),                 # ví dụ 3 của đề
    "dùng <= thay vì < (bằng limit vẫn phải giữ)": (wrong_le, [5], 5),
    "chỉ xét tổng từ gốc đến chính nút": (wrong_prefix_only, [1, 2, 3, -5, None, None, 10], 5),                # nút 3 có tổng từ gốc 4 < 5 nhưng lá 10 phía dưới làm đường dài 14
    "nút trong chỉ giữ khi cả hai con sống": (wrong_need_both, [1, 2], 3),                                      # chỉ có một con
    "quên trả None khi gốc bị xóa": (wrong_keep_root, [5], 6),
}
counters = {}
for name, (fn, tree_list, limit) in hand_cases.items():
    ok_wrong = to_list(fn(build_tree(tree_list), limit))
    right = to_list(suff_brute(build_tree(tree_list), limit))
    assert ok_wrong != right, name
    assert find_counter(fn) is not None, name            # tìm kiếm tự động cũng tìm ra phản ví dụ (nhỏ nhất)
    counters[name] = (tree_list, limit)
    print(f"{name:46} root={tree_list} limit={limit}: sai -> {ok_wrong}, đúng -> {right}")
    check_all(tree_list, limit)
print("12.3 phản ví dụ: OK")

chỉ xóa lá, không xóa nút trong                root=[1, 2, -3, -5, None, 4, None] limit=-1: sai -> [1, 2, -3, None, None, 4], đúng -> [1, None, -3, 4]


dùng <= thay vì < (bằng limit vẫn phải giữ)    root=[5] limit=5: sai -> [], đúng -> [5]


chỉ xét tổng từ gốc đến chính nút              root=[1, 2, 3, -5, None, None, 10] limit=5: sai -> [], đúng -> [1, None, 3, None, 10]
nút trong chỉ giữ khi cả hai con sống          root=[1, 2] limit=3: sai -> [], đúng -> [1, 2]


quên trả None khi gốc bị xóa                   root=[5] limit=6: sai -> [5], đúng -> []
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: nhiều hình dạng, dải giá trị và limit
rng = random.Random(4080)
for _ in range(3000):
    n = rng.randint(1, 14)
    hi = rng.choice([1, 3, 10, 100000])
    tree_list = random_tree(rng, n, -hi, hi, shape=rng.choice(["random", "random", "left", "right", "zigzag"]))
    check_all(tree_list, rng.randint(-3 * hi, 3 * hi))
print("12.4 ngẫu nhiên nhỏ: OK")

# 12.5 ngẫu nhiên vừa và lớn: các cách so với nhau (không dùng oracle)
for _ in range(60):
    n = rng.randint(100, 800)
    tree_list = random_tree(rng, n, -10, 10, shape=rng.choice(["random", "left", "right", "zigzag"]))
    check_all(tree_list, rng.randint(-40, 40), brute=False)
for _ in range(8):
    tree_list = random_tree(rng, 5000, -10 ** 5, 10 ** 5, shape=rng.choice(["random", "left", "zigzag", "right"]))
    check_all(tree_list, rng.randint(-10 ** 6, 10 ** 6), brute=False)
print("12.5 ngẫu nhiên vừa và lớn: OK")

# 12.6 tính chất: limit nhỏ hơn mọi tổng -> giữ nguyên; limit lớn hơn mọi tổng -> rỗng; kết quả là "cây con liền kề gốc"; đơn điệu theo limit; mỗi nút còn lại nằm trên đường >= limit
def count_nodes(lst):
    return len([v for v in lst if v is not None])

for _ in range(300):
    n = rng.randint(1, 14)
    tree_list = random_tree(rng, n, -10, 10, shape=rng.choice(["random", "left", "zigzag"]))
    sums = []
    st = [(build_tree(tree_list), 0)]
    while st:
        node, s = st.pop()
        s += node.val
        if not node.left and not node.right:
            sums.append(s)
        for ch in (node.left, node.right):
            if ch:
                st.append((ch, s))
    assert to_list(sol(build_tree(tree_list), min(sums))) == tree_list                  # limit = tổng nhỏ nhất: không xóa gì
    assert to_list(sol(build_tree(tree_list), max(sums) + 1)) == []                     # limit lớn hơn mọi tổng: rỗng
    counts = [count_nodes(to_list(sol(build_tree(tree_list), lim))) for lim in range(min(sums) - 1, max(sums) + 3)]
    assert counts == sorted(counts, reverse=True)                                       # limit tăng thì số nút còn lại không tăng
    res = to_list(sol(build_tree(tree_list), max(sums)))
    assert res and count_nodes(res) >= 1                                                # limit = tổng lớn nhất: còn ít nhất một đường
print("12.6 tính chất: OK")

12.4 ngẫu nhiên nhỏ: OK


12.5 ngẫu nhiên vừa và lớn: OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xóa lá, không xóa nút trong đã mất hết con

Sau khi xóa lá, một nút trong có thể trở thành **lá mới** với đường đi tổng `< limit`; nút đó cũng phải bị xóa.

In [22]:
tree_list, limit = counters["chỉ xóa lá, không xóa nút trong"]
print("root =", tree_list, "limit =", limit, "| sai ->", to_list(wrong_only_leaves(build_tree(tree_list), limit)), "| đúng ->", to_list(sol(build_tree(tree_list), limit)))
assert to_list(wrong_only_leaves(build_tree(tree_list), limit)) != to_list(sol(build_tree(tree_list), limit))

root = [1, 2, -3, -5, None, 4, None] limit = -1 | sai -> [1, 2, -3, None, None, 4] | đúng -> [1, None, -3, 4]


Vì sao sai: nút `2` chỉ nằm trên đường qua lá `-5` (đã bị xóa), nên chính nó cũng không đủ. **Đúng:** sau khi xử lý các con, nếu cả hai con đều `None` thì trả `None` cho nút trong.

### Lỗi 2 — Dùng `<=` thay vì `<`

Đề nói "tổng **nhỏ hơn hẳn** `limit`" mới bị xóa: tổng **bằng** `limit` vẫn được giữ.

In [23]:
tree_list, limit = counters["dùng <= thay vì < (bằng limit vẫn phải giữ)"]
print("root =", tree_list, "limit =", limit, "| sai ->", to_list(wrong_le(build_tree(tree_list), limit)), "| đúng ->", to_list(sol(build_tree(tree_list), limit)))
assert to_list(wrong_le(build_tree(tree_list), limit)) != to_list(sol(build_tree(tree_list), limit))

root = [5] limit = 5 | sai -> [] | đúng -> [5]


Vì sao sai: biên `=` thuộc về phía được giữ. **Đúng:** xóa lá khi `s < limit`.

### Lỗi 3 — Chỉ xét tổng từ gốc đến chính nút

Một nút trong có thể có tiền tố nhỏ nhưng các lá phía dưới làm tổng đường lớn lên (giá trị lớn ở dưới), nên **chưa** thể xóa chỉ vì tiền tố nhỏ.

In [24]:
tree_list, limit = counters["chỉ xét tổng từ gốc đến chính nút"]
print("root =", tree_list, "limit =", limit, "| sai ->", to_list(wrong_prefix_only(build_tree(tree_list), limit)), "| đúng ->", to_list(sol(build_tree(tree_list), limit)))
assert to_list(wrong_prefix_only(build_tree(tree_list), limit)) != to_list(sol(build_tree(tree_list), limit))

root = [1, 2, 3, -5, None, None, 10] limit = 5 | sai -> [] | đúng -> [1, None, 3, None, 10]


Vì sao sai: tổng đường tới **lá** mới quyết định. **Đúng:** chỉ quyết định ở lá, rồi lan lên theo "còn con sống thì giữ".

### Lỗi 4 — Chỉ giữ nút trong khi cả hai con đều sống

Nút trong có thể chỉ có **một** con; con vắng mặt không phải "con bị xóa".

In [25]:
tree_list, limit = counters["nút trong chỉ giữ khi cả hai con sống"]
print("root =", tree_list, "limit =", limit, "| sai ->", to_list(wrong_need_both(build_tree(tree_list), limit)), "| đúng ->", to_list(sol(build_tree(tree_list), limit)))
assert to_list(wrong_need_both(build_tree(tree_list), limit)) != to_list(sol(build_tree(tree_list), limit))

root = [1, 2] limit = 3 | sai -> [] | đúng -> [1, 2]


Vì sao sai: nút giữ khi **có ít nhất một** con sống (hoặc là lá đủ lớn). **Đúng:** `node if (node.left or node.right) else None`.

### Lỗi 5 — Quên trả `None` khi gốc bị xóa

Nếu cả gốc không đủ thì cây còn lại rỗng; trả `root` cũ là sai.

In [26]:
tree_list, limit = counters["quên trả None khi gốc bị xóa"]
print("root =", tree_list, "limit =", limit, "| sai ->", to_list(wrong_keep_root(build_tree(tree_list), limit)), "| đúng ->", to_list(sol(build_tree(tree_list), limit)))
assert to_list(wrong_keep_root(build_tree(tree_list), limit)) != to_list(sol(build_tree(tree_list), limit))

root = [5] limit = 6 | sai -> [5] | đúng -> []


Vì sao sai: hàm phải trả về gốc **mới** của cây; có thể là `None`. **Đúng:** `return root if keep[root] else None` (hoặc để `dfs(root, 0)` trả về trực tiếp).

### Lỗi 6 — Đệ quy quá sâu trên cây lệch

DFS đệ quy trên cây lệch `5000` tầng vượt giới hạn mặc định của Python.

In [27]:
def deep_recursive(root, limit):
    def dfs(node, s):
        if node is None:
            return None
        s += node.val
        if not node.left and not node.right:
            return None if s < limit else node
        node.left = dfs(node.left, s)
        node.right = dfs(node.right, s)
        return node if (node.left or node.right) else None
    return dfs(root, 0)

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    deep_recursive(chain_tree(5000), 0)
    print("không lỗi (không mong đợi)")
except RecursionError as e:
    print("RecursionError:", e)
finally:
    sys.setrecursionlimit(old)
print("bản BFS trên cùng cây:", to_list(sol(chain_tree(5000), 0))[:5], "...")

RecursionError: maximum recursion depth exceeded
bản BFS trên cùng cây: [1, 1, None, 1, None] ...


Vì sao sai: mỗi tầng của cây lệch là một khung đệ quy. **Đúng:** nâng `sys.setrecursionlimit` (cẩn thận) hoặc dùng duyệt lặp như Cách 3.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "xóa các nút thỏa điều kiện trên **mọi đường đi qua nó**" ⇒ quyết định ở **lá**, lan truyền lên: "nút trong còn sống khi còn con sống".
- Mẫu hàm đệ quy **trả về gốc mới**: `node.left = dfs(node.left, ...)`, `return node or None`.
- Truyền xuống **tổng tiền tố** (hoặc phần còn lại của ngưỡng), truyền lên **kết quả của cây con**.
- Trong Python tránh đệ quy sâu: BFS lưu thứ tự + xử lý ngược, hoặc ngăn xếp tự quản lý.
- "Nhỏ hơn hẳn" nghĩa là `<` (không `<=`). Luôn kiểm tra biên.
- Cùng họ: Delete Leaves With a Given Value (1325), Binary Tree Pruning (814), Path Sum (112), Maximum Path Sum (124).
- Test nên có: một nút, cây lệch, một con, tổng đúng bằng `limit`, `limit` rất nhỏ / rất lớn, giá trị âm.

## 15. Biến thể

1. **Delete Leaves With a Given Value (1325):** xóa lá có giá trị `target`, lặp lại cho đến khi không còn lá như vậy (cùng mẫu "xóa nút mất hết con").
2. **Binary Tree Pruning (814):** xóa mọi cây con chỉ chứa `0`.
3. **Xóa nút mà *tồn tại* một đường dưới `limit`** (điều kiện "tồn tại" thay vì "mọi"): quy tắc khác.
4. Bài liên quan: 814, 1325, 112.

Kiểm biến thể 1 (1325) bằng oracle "xóa lá lặp lại":

In [28]:
def remove_leaf_nodes(root, target):
    """Mẫu: xử lý hai con trước, rồi xóa nút nếu thành lá có giá trị target."""
    def dfs(node):
        if node is None:
            return None
        node.left = dfs(node.left)
        node.right = dfs(node.right)
        if not node.left and not node.right and node.val == target:
            return None
        return node
    return dfs(root)

def remove_leaf_nodes_repeat(root, target):
    """Oracle: lặp 'xóa mọi lá có giá trị target' cho tới khi không còn gì để xóa."""
    def one_pass(node):
        if node is None:
            return None
        if not node.left and not node.right:
            return None if node.val == target else node
        node.left, node.right = one_pass(node.left), one_pass(node.right)
        return node
    prev = None
    while True:
        cur = to_list(root)
        if cur == prev:
            return root
        prev = cur
        root = one_pass(root)

rng = random.Random(15)
for _ in range(1500):
    n = rng.randint(1, 12)
    tree_list = random_tree(rng, n, 0, 2, shape=rng.choice(["random", "left", "zigzag"]))
    target = rng.randint(0, 2)
    assert to_list(remove_leaf_nodes(build_tree(tree_list), target)) == to_list(remove_leaf_nodes_repeat(build_tree(tree_list), target)), (tree_list, target)
assert to_list(remove_leaf_nodes(build_tree([1, 2, 3, 2, None, 2, 4]), 2)) == [1, None, 3, None, 4]
print("biến thể Delete Leaves With a Given Value (1325): khớp oracle lặp và ví dụ của bài")

biến thể Delete Leaves With a Given Value (1325): khớp oracle lặp và ví dụ của bài


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê đường đi | Theo định nghĩa, dựng cây mới | `O(n^2)` | `O(n^2)` | Oracle |
| 2. DFS đệ quy | Truyền tổng xuống, gán lại con | `O(n)` | `O(h)` | Ngắn gọn, cần nâng giới hạn đệ quy |
| 3. BFS + xử lý ngược | `keep` từ lá lên | `O(n)` | `O(n)` | **Cách nộp chính** (không đệ quy) |
| 4. Tổng lớn nhất qua nút | `through = prefix + down` | `O(n)` | `O(n)` | Dễ chứng minh theo định nghĩa |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(to_list(build_tree([1, None, 2, 3])))
print(to_list(Solution().sufficientSubset(build_tree([1, 2, 3]), 3)))
print(to_list(Solution().sufficientSubset(build_tree([1, 2, 3]), 5)))
```

In [29]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # danh sách xuất ra từ cây [1, None, 2, 3]
my_guess_2 = None      # kết quả với limit = 3
my_guess_3 = None      # kết quả với limit = 5

### Bài 2 — Tính tay (dễ-vừa)

`root = [10, 5, -3, 3, 2, None, 11]`, `limit = 18`. Liệt kê tổng các đường gốc-lá, nút nào bị xóa, và ghi danh sách kết quả.

In [30]:
# Bài 2: điền danh sách tổng các đường (theo thứ tự lá trái sang phải) và kết quả
hand_sums, hand_answer = None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DFS đệ quy.

In [31]:
def exercise_3(root, limit):
    def dfs(node, s):
        if node is None:
            return None
        s += node.val
        if not node.left and not node.right:
            # TODO 1: trả về None nếu s < limit, ngược lại trả về node
            pass
        # TODO 2: gán node.left = dfs(node.left, s) và node.right = dfs(node.right, s)
        # TODO 3: trả về node nếu còn ít nhất một con, ngược lại None
        pass
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old, 20000))
    try:
        return dfs(root, 0)
    finally:
        sys.setrecursionlimit(old)


# for tree, limit, expected in examples:
#     assert to_list(exercise_3(build_tree(tree), limit)) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(root, limit)` bằng **BFS + xử lý ngược** (không đệ quy).

In [32]:
def exercise_4(root, limit):
    # TODO
    pass


# for tree, limit, expected in examples:
#     assert to_list(exercise_4(build_tree(tree), limit)) == expected

### Bài 5 — Mini challenge: đếm số nút bị xóa (khó)

Viết `exercise_5(root, limit)` trả về **số nút bị xóa** khi áp dụng bài toán này, **không sửa cây**. Ví dụ với ví dụ 3 của đề (`[1,2,-3,-5,null,4,null]`, `limit = -1`): cây ban đầu có `5` nút, còn lại `3` nút ⇒ trả `2`.

Gợi ý: dùng `through(v) = prefix(v) + down(v)` (Cách 4); nút bị xóa là nút có `through < limit` (đếm tất cả nút của các cây con bị cắt, vì chúng đều có `through < limit`).

In [33]:
def exercise_5(root, limit) -> int:
    # TODO
    pass


# assert exercise_5(build_tree([1, 2, -3, -5, None, 4, None]), -1) == 2
# assert exercise_5(build_tree([5]), 6) == 1
# assert exercise_5(build_tree([5]), 5) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: danh sách theo tầng liệt kê hai con của mỗi nút không rỗng; `null` là vắng mặt.
- Mức 2: `[1, None, 2, 3]`: gốc `1` không có con trái, con phải `2`; `2` có con trái `3`.
- Mức 3: `[1, None, 2, 3]` (xuất ra đúng như vậy); cây `[1,2,3]` với `limit = 3` có hai đường `3` và `4` đều `>= 3` nên giữ nguyên `[1, 2, 3]`; `limit = 5` xóa hết `[]`.

**Bài 2**
- Mức 1: các lá là `3`, `2`, `11` (từ trái sang phải).
- Mức 2: đường `10 + 5 + 3 = 18`, `10 + 5 + 2 = 17`, `10 - 3 + 11 = 18`.
- Mức 3: tổng `[18, 17, 18]`; lá `2` có `17 < 18` bị xóa; nút `5` còn con `3` sống; kết quả `[10, 5, -3, 3, None, None, 11]`.

**Bài 3**
- Mức 1: lá: `return None if s < limit else node`.
- Mức 2: `node.left = dfs(node.left, s)`; `node.right = dfs(node.right, s)`.
- Mức 3: `return node if (node.left or node.right) else None`.

**Bài 4**
- Mức 1: BFS lưu `nodes` và `prefix`.
- Mức 2: duyệt ngược; lá: `keep = prefix >= limit`; nút trong: cắt con có `keep` sai, rồi `keep = còn con`.
- Mức 3: `return root if keep[root] else None`.

**Bài 5**
- Mức 1: dựng `prefix` và `down` giống Cách 4, nhưng không cắt.
- Mức 2: nút có `through < limit` là nút bị xóa.
- Mức 3: `sum(1 for v in nodes if through[v] < limit)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [34]:
r1 = to_list(build_tree([1, None, 2, 3]))
r2 = to_list(Solution().sufficientSubset(build_tree([1, 2, 3]), 3))
r3 = to_list(Solution().sufficientSubset(build_tree([1, 2, 3]), 5))
print(r1, r2, r3)
assert r1 == [1, None, 2, 3]
assert r2 == [1, 2, 3]
assert r3 == []

[1, None, 2, 3] [1, 2, 3] []


### Đáp án Bài 2

Các lá `3`, `2`, `11`: tổng `[18, 17, 18]`. Lá `2` (tổng `17 < 18`) bị xóa; nút `5` giữ nhờ lá `3`; nhánh `-3` giữ nhờ lá `11`. Kết quả `[10, 5, -3, 3, None, None, 11]`.

In [35]:
hand_sums_solution, hand_answer_solution = [18, 17, 18], [10, 5, -3, 3, None, None, 11]
tree_list = [10, 5, -3, 3, 2, None, 11]
assert sorted(hand_sums_solution) == sorted(sum(p) for p in all_root_leaf_paths(build_tree(tree_list)))
assert hand_answer_solution == to_list(sol(build_tree(tree_list), 18)) == to_list(suff_brute(build_tree(tree_list), 18))
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [36]:
def exercise_3_solution(root, limit):
    def dfs(node, s):
        if node is None:
            return None
        s += node.val
        if not node.left and not node.right:
            return None if s < limit else node
        node.left = dfs(node.left, s)
        node.right = dfs(node.right, s)
        return node if (node.left or node.right) else None
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old, 20000))
    try:
        return dfs(root, 0)
    finally:
        sys.setrecursionlimit(old)

for tree, limit, expected in examples:
    assert to_list(exercise_3_solution(build_tree(tree), limit)) == expected
rng = random.Random(3)
for _ in range(500):
    tree_list = random_tree(rng, rng.randint(1, 12), -5, 8)
    limit = rng.randint(-8, 12)
    assert to_list(exercise_3_solution(build_tree(tree_list), limit)) == to_list(suff_brute(build_tree(tree_list), limit))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [37]:
def exercise_4_solution(root, limit):
    nodes, prefix = [root], [root.val]
    i = 0
    while i < len(nodes):
        node = nodes[i]
        for ch in (node.left, node.right):
            if ch:
                nodes.append(ch)
                prefix.append(prefix[i] + ch.val)
        i += 1
    keep = {}
    for i in range(len(nodes) - 1, -1, -1):
        node = nodes[i]
        if not node.left and not node.right:
            keep[node] = prefix[i] >= limit
        else:
            if node.left and not keep[node.left]:
                node.left = None
            if node.right and not keep[node.right]:
                node.right = None
            keep[node] = bool(node.left or node.right)
    return root if keep[root] else None

for tree, limit, expected in examples:
    assert to_list(exercise_4_solution(build_tree(tree), limit)) == expected
rng = random.Random(4)
for _ in range(500):
    tree_list = random_tree(rng, rng.randint(1, 12), -5, 8)
    limit = rng.randint(-8, 12)
    assert to_list(exercise_4_solution(build_tree(tree_list), limit)) == to_list(suff_brute(build_tree(tree_list), limit))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [38]:
def exercise_5_solution(root, limit) -> int:
    nodes, prefix = [root], [root.val]
    i = 0
    while i < len(nodes):
        node = nodes[i]
        for ch in (node.left, node.right):
            if ch:
                nodes.append(ch)
                prefix.append(prefix[i] + ch.val)
        i += 1
    down = {}
    for node in reversed(nodes):
        gains = [c.val + down[c] for c in (node.left, node.right) if c]
        down[node] = max(gains) if gains else 0
    return sum(1 for i, node in enumerate(nodes) if prefix[i] + down[node] < limit)

assert exercise_5_solution(build_tree([1, 2, -3, -5, None, 4, None]), -1) == 2
assert exercise_5_solution(build_tree([5]), 6) == 1
assert exercise_5_solution(build_tree([5]), 5) == 0
rng = random.Random(5)
for _ in range(500):
    tree_list = random_tree(rng, rng.randint(1, 12), -5, 8)
    limit = rng.randint(-8, 12)
    before = count_nodes(tree_list)
    after = count_nodes(to_list(suff_brute(build_tree(tree_list), limit)))
    assert exercise_5_solution(build_tree(tree_list), limit) == before - after, (tree_list, limit)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DFS đệ quy trả về gốc mới, truyền tổng xuống)

```python
class Solution:
    def sufficientSubset(self, root: Optional[TreeNode], limit: int) -> Optional[TreeNode]:
        def dfs(node, s):
            if node is None:
                return None
            s += node.val                                   # tổng từ gốc tới node
            if not node.left and not node.right:            # lá: giữ nếu tổng >= limit
                return None if s < limit else node
            node.left = dfs(node.left, s)
            node.right = dfs(node.right, s)
            return node if (node.left or node.right) else None
        return dfs(root, 0)
```

(Với cây lệch sâu `5000`: nâng `sys.setrecursionlimit` hoặc dùng BFS + xử lý ngược như Cách 3.)

Công thức nhớ nhanh: **quyết định ở lá (`tổng < limit` thì xóa); nút trong bị xóa khi cả hai con (nếu có) đã bị xóa; trả về gốc mới (có thể `None`)**.

### Checklist trước khi submit

- [ ] Lá bị xóa khi tổng `< limit` (nghiêm ngặt), tổng bằng `limit` vẫn giữ?
- [ ] Nút trong bị xóa khi **không còn con nào** sống (nút chỉ có một con vẫn xử lý đúng)?
- [ ] Hàm trả về gốc mới, có thể là `None`, và con được **gán lại** từ kết quả?
- [ ] Không phụ thuộc đệ quy sâu (hoặc đã nâng giới hạn) với cây lệch `5000`?
- [ ] Khi test: dựng cây mới cho mỗi lần gọi vì hàm sửa cây tại chỗ?
- [ ] Đã thử: một nút, cây lệch, `limit` rất nhỏ / rất lớn, giá trị âm?